## Clean Data

In [ ]:
# This notebook demonstrates how to extract time-series data from a cloud database 
# and clean it using Local Outlier Factor (LOF) for anomaly detection.

#Mock Packages

In [ ]:
import itertools
import json
import mlflow
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from mlflow.models import infer_signature
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# Initialize tracking client
ml_client = get_cloud_ml_client()
set_mlflow_tracking_uri(ml_client)

In [ ]:
%reload_ext Kqlmagic
%kql AzureDataExplorer://code;cluster='YOUR_CLUSTER_NAME.westeurope';database='YOUR_DATABASE_NAME' -try-azcli-login

## Preprocessing - Retrieve your Data (case scenario from ADX)

Install KQL in the environment!

In [ ]:
%%kql
data_processed << ADX_TABLE_NAME
| where Timestamp between (datetime(2000-00-00) .. datetime(2000-00-00))
| project-keep Timestamp,
    ['EXACT SENSOR NAME']
| order by Timestamp asc
| extend Timestamp = tostring(Timestamp)

# Preprocess

In [ ]:
df_processed = data_processed.to_dataframe()
df_processed["Timestamp"] = pd.to_datetime(df_processed["Timestamp"])
df_processed = df_processed.set_index("Timestamp")
df_processed.index = df_processed.index.tz_localize(None)

df_processed.columns = df_processed.columns.str.replace("_plus_", "+")
df_processed = df_processed[~df_processed.index.duplicated()]
df_processed.dropna(inplace=True)
df_processed.columns

## Eliminate Outliers

In [ ]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor

 
contamination = 0.0001 #If you have 10,000 values in a column and contamination=0.001, the LOF "expects to find" about 10 outliers.
max_neighbors = 50 #neighbor region

# Create cleaned copy and outlier dictionary
df_processed_cleaned = df_processed.copy()
outliers_dict = {}

# Detect outliers for each column
for column in df_processed.columns:
    valid_data = df_processed[column].dropna()
    if len(valid_data) > 5:
        n_neighbors =  max_neighbors
        #lof = LocalOutlierFactor(n_neighbors=n_neighbors)
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
        lof_pred = lof.fit_predict(valid_data.values.reshape(-1, 1))

        outlier_idx = valid_data.index[lof_pred == -1]
        outliers_dict[column] = outlier_idx
        df_processed_cleaned.loc[outlier_idx, column] = np.nan
    else:
        outliers_dict[column] = []

print("Original shape:", df_processed.dropna().shape, " | Cleaned shape:", df_processed_cleaned.dropna().shape)


# Start your correlation analysis - Best Candidates for Trainning

In [ ]:
corr_matrix = df_merged.corr()
sns.heatmap(corr_matrix, cmap="crest")

corr_matrix.to_csv("corr_matrix.csv")

# Dictionary to store results for each column
top5_results = {}

# Loop through the first 5 columns
for col_name in corr_matrix.columns[:]:
    # Select the current column
    column = corr_matrix[col_name].xlxs

    # Create a DataFrame with original and absolute values
    df_temp = pd.DataFrame({
        "original_value": column,
        "absolute_value": column.abs()
    })

    # Sort by absolute value
    df_temp = df_temp.sort_values(by="absolute_value", ascending=False)

    # Take top 5 rows
    top5 = df_temp.head(6).copy()

    # Add a flag for whether absolute value was applied
    top5["abs_applied"] = top5["original_value"] < 0

    # Store in the dictionary
    top5_results[col_name] = top5[["original_value", "abs_applied"]]

    # Optional: print each result
    print(f"\nTop 5 absolute values for column: {col_name}")
    print(top5_results[col_name])


## Correlation Matrix

In [ ]:
#save view
# Save all top 5 results into a single text file
with open("top_5_absolute_values.txt", "w") as f:
    for col_name, df in top5_results.items():
        f.write(f"Top 5 absolute values for column: {col_name}\n")
        f.write(df.to_string())
        f.write("\n\n" + "-"*50 + "\n\n")

#Filter Matrix
#Parser:
#read csv document, find the values and convert the rest of the correlation matrix to NAN 
import pandas as pd

def parse_top5_to_matrix(file_path):
    data = {}

    with open(file_path, "r") as f:
        content = f.read()

   
    blocks = content.strip().split("Top 5 absolute values for column:")
    
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        
        lines = block.splitlines()
        target = lines[0].strip()
        if target not in data:
            data[target] = {}
        
        
        for line in lines[2:]:  
            parts = line.split()
            if len(parts) >= 2:
                var_name = parts[0]
                corr_value = float(parts[1])
                # Guardar en data
                data[target][var_name] = corr_value

   
    df = pd.DataFrame(data).sort_index().sort_index(axis=1)

   
    df = df.applymap(lambda x: None if x == 1.0 else x)

    
    df = df.dropna(how='all')

    return df


corr_matrix_filtered = parse_top5_to_matrix("top_5_absolute_values.txt")


print(corr_matrix_filtered.head())
corr_matrix_filtered

## Linear Regression

In [ ]:
#Linear Regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import itertools
import numpy as np
import pandas as pd

results = []
max_inputs = 5

# Define the column groups and their names
target_groups = {
    "west_LR": columns_west,
    "ost_LR": columns_ost,
    "mitte_LR": columns_mitte
}

for group_name, target_columns in target_groups.items():
    results = []  # Reset results for each group

    for target in corr_matrix_filtered.columns:
        if target in target_columns:
            inputs = corr_matrix_filtered[target].dropna()
            inputs = inputs[inputs.index != target].index.tolist()
            df_input = df_merged.dropna(how="any").copy()

            early_stop = False
            input_3_flag = False
            print(f"\n[Processing] Target Sensor: {target}")

            for i in range(1, len(inputs) + 1):
                if i > max_inputs and input_3_flag:
                    print(
                        f"[Stop beyond {max_inputs}-inputs] No good results with {max_inputs} inputs for {target}. Ending search."
                    )
                    break  # Stop processing higher than max_inputs combinations

                combo_stop = False
                result_found = False

                for combo in itertools.combinations(inputs, i):
                    x = df_input[list(combo)]
                    y = df_input[target]

                    # Drop rows with NaN values in the selected columns
                    x = x.dropna()
                    y = y[x.index]
                    x_train, x_test, y_train, y_test = train_test_split(
                        x, y, test_size=0.2, random_state=11
                    )

                    model = LinearRegression()
                    model.fit(x_train, y_train)

                    y_pred = model.predict(x_test)
                    score_r2 = r2_score(y_test, y_pred)
                    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

                    # build row dict with weights
                    row = {
                        "target": target,
                        "inputs": combo,
                        "r2_score": score_r2,
                        "rmse": rmse,
                    }

                    # add weights for each input in the combo
                    for feature, coef in zip(combo, model.coef_):
                        row[f"weight_{feature}"] = coef

                    results.append(row)

                    print(f"Target: {target}, Inputs: {combo}, R2 Score: {score_r2}, RMSE: {rmse}")

                    if score_r2 >= 0.985 and i >= 3:
                        print(
                            f"Early stopping for target {target} with inputs {combo}, R2 score {score_r2}, RMSE: {rmse}"
                        )
                        early_stop = True
                        combo_stop = True
                        break
                    if score_r2 > 0.91:
                        result_found = True
                if combo_stop:
                    break

                if i == max_inputs:
                    print(
                        f"[Stop triggered at {max_inputs}-input level] No combinations gave acceptable R2 for {target}."
                    )
                    input_3_flag = True

    # Save results for each group
    results_df_v2 = pd.DataFrame(results)
    results_df_v2.to_csv(f"MODEL_linear_regression_v2_{group_name}.csv", index=False)


## XGBoost

In [ ]:
#XGBOOST
# XGBoost regression search with groups

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import itertools
import numpy as np
import pandas as pd

max_inputs = 5

# Define target groups
target_groups = {
    "west_XGB": columns_west,
    "ost_XGB": columns_ost,
    "mitte_XGB": columns_mitte
}

for group_name, target_columns in target_groups.items():
    results = []  # Reset results for each group

    for target in corr_matrix_filtered:
        if target in target_columns:
            inputs = corr_matrix_filtered[target].dropna()
            inputs = inputs[inputs.index != target].index.tolist()
            df_input = df_merged.dropna(how="any").copy()

            early_stop = False
            input_3_flag = False
            print(f"\n[Processing] Target Sensor: {target} (Group: {group_name})")

            for i in range(1, len(inputs) + 1):
                if i > max_inputs and input_3_flag:
                    print(
                        f"[Stop beyond {max_inputs}-inputs] No good results with {max_inputs} inputs for {target}. Ending search."
                    )
                    break  # Stop processing combinations beyond max_inputs

                combo_stop = False
                result_found = False

                for combo in itertools.combinations(inputs, i):
                    x = df_input[list(combo)]
                    y = df_input[target]

                    # Drop rows with NaN values in the selected columns
                    x = x.dropna()
                    y = y[x.index]
                    train_size = int(len(x) * 0.8)
                    x_train, x_test = x[:train_size], x[train_size:]
                    y_train, y_test = y[:train_size], y[train_size:]

                    # model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5)
                    # Train XGBoost model (simplified baseline parameters)
                    model = XGBRegressor(
                        n_estimators=300,     # number of trees
                        learning_rate=0.05,   # step size shrinkage
                        max_depth=3,          # tree depth (keep shallow to avoid overfit)
                        subsample=0.8,        # sample fraction of rows
                        colsample_bytree=0.8, # sample fraction of features
                        random_state=11,
                        n_jobs=-1
                    )
                    model.fit(x_train, y_train)

                    y_pred = model.predict(x_test)
                    score_r2 = r2_score(y_test, y_pred)
                    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                    print(
                        f"Target: {target}, Inputs: {combo}, R2 Score: {score_r2}, RMSE: {rmse}"
                    )
                    results.append(
                        {
                            "y_variable": target,
                            "x_variables": combo,
                            "r2_score": score_r2,
                            "rmse": rmse,
                        }
                    )

                    if score_r2 >= 0.98:
                        print(
                            f"Early stopping for target {target} with inputs {combo}, R2 score {score_r2}, RMSE: {rmse}"
                        )
                        early_stop = True
                        combo_stop = True
                        break
                    if score_r2 > 0.91:
                        result_found = True
                if combo_stop:
                    break

                if i == max_inputs:
                    print(
                        f"[Stop triggered at {max_inputs}-input level] No combinations gave acceptable R2 for {target}."
                    )
                    input_3_flag = True

    # Save results for the current group
    results_df = pd.DataFrame(results)
    #results_df.to_csv(f"B443_regression_results_{group_name}.csv", index=False)
    results_df.to_csv(f"MODELNAME{group_name}.csv", index=False)

## MLP

In [ ]:
# MLP
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import itertools
import numpy as np
import pandas as pd

# Using MLPRegressor for the same task
max_inputs = 2

# Define the column groups and their names
target_groups = {
    "mitte_MLP": columns_mitte,
    "ost_MLP": columns_ost,
    "west_MLP": columns_west  # Adjust these names/variables as needed
}

for group_name, target_columns in target_groups.items():
    results = []  # Reset results for each group

    for target in corr_matrix_filtered.columns:
        if target in target_columns:
            inputs = corr_matrix_filtered[target].dropna()
            inputs = inputs[inputs.index != target].index.tolist()
            df_input = df_merged.dropna(how="any").copy()

            early_stop = False
            input_3_flag = False
            print(f"\n[Processing] Target Sensor: {target} (Group: {group_name})")

            for i in range(1, len(inputs) + 1):
                if i > max_inputs and input_3_flag:
                    print(
                        f"[Stop beyond {max_inputs}-inputs] No good results with {max_inputs} inputs for {target}. Ending search."
                    )
                    break  # Stop processing higher than max_inputs combinations

                combo_stop = False
                result_found = False
                combinations = list(set(itertools.combinations(inputs, i)))

                for combo in combinations:
                    x = df_input[list(combo)]
                    y = df_input[target]

                    # Drop rows with NaN values in the selected columns
                    x = x.dropna()
                    y = y[x.index]
                    train_size = int(len(x) * 0.8)
                    x_train, x_test = x[:train_size], x[train_size:]
                    y_train, y_test = y[:train_size], y[train_size:]

                    model = MLPRegressor(
                        hidden_layer_sizes=(50, 25),
                        learning_rate="adaptive",
                        max_iter=250,
                        random_state=7,
                        verbose=False,
                        early_stopping=True,
                        validation_fraction=0.2,
                        n_iter_no_change=10,
                    )
                    model.fit(x_train, y_train)

                    y_pred = model.predict(x_test)
                    score_r2 = r2_score(y_test, y_pred)
                    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                    print(
                        f"Target: {target}, Inputs: {combo}, R2 Score: {score_r2}, RMSE: {rmse}"
                    )
                    results.append(
                        {
                            "target": target,
                            "inputs": combo,
                            "r2_score": score_r2,
                            "rmse": rmse,
                        }
                    )

                    if score_r2 >= 0.98:
                        print(
                            f"Early stopping for target {target} with inputs {combo}, R2 score {score_r2}, RMSE: {rmse}"
                        )
                        early_stop = True
                        combo_stop = True
                        break
                    if score_r2 > 0.91:
                        result_found = True
                if combo_stop:
                    break

                if i == max_inputs:
                    print(
                        f"[Stop triggered at {max_inputs}-input level] No combinations gave acceptable R2 for {target}."
                    )
                    input_3_flag = True

    # Save results for this group
    results_df = pd.DataFrame(results)
    #results_df.to_csv(f"kbb_regression_results_{group_name}.csv", index=False)
    results_df.to_csv(f"MLP-MODEL{group_name}.csv", index=False)


## Statistical Limit Values

In [ ]:
import pandas as pd

# File path
file_path = r"EXCELFILE"

# Function to filter rows with 'x' in the 'x' column
def load_filtered_sheet(sheet_name):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    
    # Check for required columns
    assert {"target", "inputs", "r2_score"}.issubset(df.columns), (
        f"Sheet {sheet_name} does not have the required columns"
    )
    
    # Filter rows with 'x' in the 'x' column
    df_filtered = df[df["x"].astype(str).str.lower() == "x"]
    
    # Reset index and sort by 'target'
    df_filtered = df_filtered.reset_index(drop=True).sort_values(by="target")
    
    return df_filtered

# Load each filtered sheet
df_mitte = load_filtered_sheet("mitte_LR")
df_ost = load_filtered_sheet("ost_LR")
df_west = load_filtered_sheet("west_LR")

# Concatenate all sheets
result = pd.concat([df_mitte, df_ost, df_west], axis=0)
result = result.reset_index(drop=True)

# Rename columns
result = result.rename(columns={"target": "y_variable", "inputs": "x_variables"})

# Add additional columns
result["Beziehung"] = "Linear"
result["Modell"] = "Linear Regression"
result["Lag"] = 0

# Save to CSV
result.to_csv("x-y_mappen_new.csv", index=False)

# Show the result
result

# Train again the model with the dictionary
split_idx = int(len(df_merged) * 0.2)


df_test = df_merged.iloc[:split_idx].copy()    # 20%
df_train = df_merged.iloc[split_idx:].copy()   # 80%

from mlflow.models import infer_signature

model = MultipleLinearRegression(input_dict)
metric_scores = model.fit(df_train, df_test, metrics={"r2_score": r2_score})
df_metric_scores = pd.DataFrame(metric_scores)

df_train_predict = model._predict(df_train)
df_test_predict = model._predict(df_test)

# infer signature
signature = infer_signature(df_train, df_train_predict)
print(signature)

df_in = df_merged[columns].copy()
df_in.shape,df_merged.shape





In [ ]:
import quantile_exp_tail #Train your model
import numpy as np

# df_predicted = predict(df=df_test, endpoint_name="MODEL_linear-regression")
df_in.dropna(inplace=True, how="any")
df_predicted = model._predict(df_in)

#Calculate Difference

df_diff = (df_merged[columns] - df_predicted).abs()
threshold_types = [
    "n3_sigma_quaddiff",
    "n6_sigma_quaddiff",
    "expquantile_quaddiff",
]

thresholds = {}

for col in df_diff.columns:
    threshold_name = f"{col}"
    quad_diff = df_diff[col] ** 2
    std_quad = quad_diff.std()
    
    thresholds[threshold_name] = {}
    
    # n=3 * std of quad_diff
    thresholds[threshold_name]["n3_sigma_quaddiff"] = 3 * std_quad

    # n=6 * std of quad_diff
    thresholds[threshold_name]["n6_sigma_quaddiff"] = 6 * std_quad

    # expquantile of quad_diff
    thresholds[threshold_name]["expquantile_quaddiff"] = quantile_exp_tail(
        np.array(quad_diff), p_lim=0.9999, p_th=0.99
    )

df_thresholds = pd.DataFrame.from_dict(thresholds, orient="index").T
df_thresholds.columns = df_thresholds.columns.str.replace("_plus_", "+")
df_thresholds.index.name = "Method"

df_thresholds.to_csv("SHEET_NAME.csv", index=True)
df_thresholds

## Plot

In [ ]:


#Plot Schwellenwert

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_with_thresholds(
    df_merged,
    df_predicted,
    df_thresholds,
    column_name,
    threshold_type="stdabw_absdiff",
    start_date="2024-04-01"  # <-- Added start date parameter
):
    # Filter data from start_date onwards
    df_merged = df_merged.loc[df_merged.index >= start_date]
    df_predicted = df_predicted.loc[df_predicted.index >= start_date]

    # Actual and predicted values
    y_actual = df_merged[column_name].dropna()
    y_pred = df_predicted[column_name].reindex(y_actual.index)

    # Calculate difference (squared absolute)
    diff = (y_actual - y_pred).abs() ** 2

    # Get the threshold value for this column and type
    threshold_value = df_thresholds.loc[threshold_type, column_name]

    # Create subplots
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.05,
        subplot_titles=(f"{column_name} - Actual vs Predicted", "Squared Difference")
    )

    # Top: Actual vs Predicted
    fig.add_trace(
        go.Scatter(x=y_actual.index, y=y_actual, mode="lines", name="Actual"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=y_pred.index, y=y_pred, mode="lines", name="Predicted"),
        row=1, col=1
    )

    # Bottom: Difference
    fig.add_trace(
        go.Scatter(x=diff.index, y=diff, mode="lines", name="Difference"),
        row=2, col=1
    )

    # Add threshold line
    fig.add_trace(
        go.Scatter(
            x=diff.index,
            y=[threshold_value] * len(diff),
            mode="lines",
            line=dict(dash="dash", color="red"),
            name=f"Threshold ({threshold_type})"
        ),
        row=2, col=1
    )

    # Layout
    fig.update_layout(
        height=600,
        title=f"Comparison and Difference for {column_name} (Threshold: {threshold_type})",
        showlegend=True
    )

    fig.show()

## Create ML Experiment

In [ ]:
# Example usage:
col_name = "TARGET SENSOR"
plot_with_thresholds(
    df_merged,
    df_predicted,
    df_thresholds,
    col_name,
    threshold_type="expquantile_quaddiff",
    start_date="2024-04-01"  # <-- Start plotting from April 1st
)


## Create an Experiment in ML

model_name = "HERE COMES THE MODEL NAME"
experiment_name = f"{model_name}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
experiment_id = get_or_create_experiment(experiment_name=experiment_name)

signature

with mlflow.start_run(experiment_id=experiment_id):
    mlflow.log_metric(
        "r2_score_train", value=df_metric_scores.loc["r2_score_train"].median()
    )
    mlflow.log_metric("r2_score_test", df_metric_scores.loc["r2_score_test"].median())

    mlflow.pyfunc.log_model(
        python_model=model,
        artifact_path="model",
        code_paths=["../libs/"],
        signature=signature,
        input_example=df_train.head(),
    )

# register model
df_runs = mlflow.search_runs(experiment_ids=[experiment_id])
df_runs

model_name = "HERE COMES THE MODEL NAME"
azureml_model_info = ml_client.models.get(model_name, label="latest")
azureml_model_info

model = mlflow.pyfunc.load_model(
    azureml_model_info.properties.get("mlflow.modelSourceUri")
)

df_predicted_ML = model.predict(df_in)
df_predicted_ML